# Stage 0A — Reconstruct sample-ID alignment from original files

**Source inputs:** `data/source/ibd_datasets.rar` (eight unaligned CSVs) and `data/source/ibd_datasets.RData` (original R data.frames with count-table row names). **Frozen reference:** `data/validated/ibd_dataset_sample_aligned.zip` (SHA-256 pinned).

The original CSV count tables are **missing patient IDs**. Their RData counterpart retains the IDs as row names, so we match counts to `desc_d#.csv` by these IDs and **never by row position**. This is an identity correction **not normalization or filtering**. The original immutable dataset is never overwritten. The narrow RDX3/XDR parser is in `src/cmps355/rdata_xdr_reader.py` and refuses unsupported R serialization types.

In [ ]:
from pathlib import Path
import os, sys, json
from datetime import datetime, timezone
import pandas as pd

candidates = [os.environ.get("CMPS355_ROOT"), Path.cwd(), Path.cwd().parent,
              "/content/drive/MyDrive/CMPS355_Research"]
ROOT = next((Path(x).resolve() for x in candidates if x and
             (Path(x)/"data/validated/ibd_dataset_sample_aligned.zip").is_file() and
             (Path(x)/"src/cmps355").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Set CMPS355_ROOT to the project root containing data/validated and src/cmps355")
sys.path.insert(0,str(ROOT / "src"))
RUN_ID = os.environ.get("CMPS355_RUN_ID") or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
print("PROJECT_ROOT:", ROOT)
print("RUN_ID:", RUN_ID)


In [ ]:
from cmps355.original_data_alignment import run_from_root
from hashlib import sha256
INPUT_RAR=ROOT/'data/source/ibd_datasets.rar'
INPUT_RDATA=ROOT/'data/source/ibd_datasets.RData'
print('Original RAR bytes:',INPUT_RAR.stat().st_size)
print('Original RData bytes:',INPUT_RDATA.stat().st_size)
for path in (INPUT_RAR,INPUT_RDATA):
    print(path.name, sha256(path.read_bytes()).hexdigest())

In [ ]:
RUN_DIR = ROOT/'runs/source_alignment'/RUN_ID
report = run_from_root(ROOT,RUN_DIR,save_rebuilt_zip=False)
assert report['status'].startswith('PASS_')
assert report['total_samples']==689 and report['requiring_reorder']==670
print('SOURCE_ALIGNMENT_STATUS:', report['status'])
print('Canonical ZIP preserved; rebuilt ZIP was verified semantically and removed to avoid duplication.')

In [ ]:
summary=pd.read_csv(RUN_DIR/'cohort_alignment_summary.csv')
display(summary)
print('Rows already matched:',summary['already_aligned'].sum())
print('Rows fixed by ID:',summary['reordered'].sum())
assert summary['already_aligned'].sum()==19
assert summary['reordered'].sum()==670
print('New auditable run:',RUN_DIR)

## How to interpret this check

The script verifies **every original CSV count against the corresponding RData data.frame**, all reconstructed sample-to-gene values against the accepted corrected ZIP, and original metadata/labels against the RData and canonical metadata. The only permitted modification at this stage is **reordering and attaching sample IDs**. It does not claim to reproduce upstream FASTQ quality control, STAR alignment, transcript quantification, or the publication's sequencing pipeline.

The machine-readable row-level mapping is retained privately in `source_row_alignment_audit.csv`; the aggregate cohort table is suitable for reporting. The acceptance reference is a content SHA-256 plus full semantic equality; a regenerated ZIP can have different byte-level hashes due to CSV formatting or ZIP metadata.